# PANDA — EfficientNet-B0 baseline (inference)



In [ ]:
import os
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import timm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    import openslide
    HAVE_OPENSLIDE = True
except Exception:
    HAVE_OPENSLIDE = False
print('device:', device, '| timm:', timm.__version__, '| openslide:', HAVE_OPENSLIDE)

## Configuration


In [ ]:
data_dir   = '/kaggle/input/competitions/prostate-cancer-grade-assessment'
MODEL_NAME = 'effnetb0_36x256_fold0'                 # <-- same as training
MODEL_DIR  = '/kaggle/input/datasets/shashaboii/panda-effnetb0-fold0'      # <-- dataset from training output
BACKBONE   = 'efficientnet_b0'

out_dim     = 5
LEVEL       = 1
tile_size   = 256
image_size  = 256
n_tiles     = 36
batch_size  = 8
num_workers = 4

df_test  = pd.read_csv(os.path.join(data_dir, 'test.csv'))
df_train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_dir = os.path.join(data_dir, 'test_images')
is_test  = os.path.exists(test_dir)
image_folder = test_dir if is_test else os.path.join(data_dir, 'train_images')
df = df_test if is_test else df_train.loc[:100].copy()
print('is_test:', is_test, '| slides:', len(df))

In [ ]:
def read_slide(path, level=LEVEL):
    if HAVE_OPENSLIDE:
        s = openslide.OpenSlide(path)
        lv = min(level, s.level_count - 1)
        img = s.read_region((0, 0), lv, s.level_dimensions[lv]).convert('RGB')
        s.close()
        return np.asarray(img)
    import tifffile
    with tifffile.TiffFile(path) as tif:
        ser = tif.series[0]
        arr = (ser.levels[min(level, len(ser.levels) - 1)].asarray()
               if len(ser.levels) > 1 else ser.asarray())
    arr = np.asarray(arr)
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, -1)
    return arr[..., :3]

## Model


In [ ]:
class enetv2(nn.Module):
    def __init__(self, backbone=BACKBONE, out_dim=out_dim, pretrained=False):
        super().__init__()
        self.enet = timm.create_model(backbone, pretrained=pretrained,
                                      num_classes=0, global_pool='avg')
        self.myfc = nn.Linear(self.enet.num_features, out_dim)
    def forward(self, x):
        return self.myfc(self.enet(x))

model = enetv2(pretrained=False)
ckpt = os.path.join(MODEL_DIR, f'{MODEL_NAME}_best.pth')
model.load_state_dict(torch.load(ckpt, map_location='cpu'))
model.eval().to(device)
print('loaded:', ckpt)

## Dataset


In [ ]:
def get_tiles(img, mode=0):
    h, w, _ = img.shape
    pad_h = (tile_size - h % tile_size) % tile_size + ((tile_size * mode) // 2)
    pad_w = (tile_size - w % tile_size) % tile_size + ((tile_size * mode) // 2)
    img2 = np.pad(img, [[pad_h // 2, pad_h - pad_h // 2],
                        [pad_w // 2, pad_w - pad_w // 2], [0, 0]], constant_values=255)
    img3 = img2.reshape(img2.shape[0] // tile_size, tile_size,
                        img2.shape[1] // tile_size, tile_size, 3)
    img3 = img3.transpose(0, 2, 1, 3, 4).reshape(-1, tile_size, tile_size, 3)
    if len(img3) < n_tiles:
        img3 = np.pad(img3, [[0, n_tiles - len(img3)], [0, 0], [0, 0], [0, 0]], constant_values=255)
    idxs = np.argsort(img3.reshape(img3.shape[0], -1).sum(-1))[:n_tiles]
    img3 = img3[idxs]
    return [{'img': img3[i], 'idx': i} for i in range(len(img3))]


class PANDADataset(Dataset):
    def __init__(self, df, image_size, n_tiles=n_tiles, tile_mode=0):
        self.df = df.reset_index(drop=True); self.image_size = image_size
        self.n_tiles = n_tiles; self.tile_mode = tile_mode
    def __len__(self):
        return self.df.shape[0]
    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = read_slide(os.path.join(image_folder, f'{row.image_id}.tiff'))
        tiles = get_tiles(image, self.tile_mode)
        n_row = int(np.sqrt(self.n_tiles))
        out = np.zeros((image_size * n_row, image_size * n_row, 3))
        for hh in range(n_row):
            for ww in range(n_row):
                i = hh * n_row + ww
                tile = tiles[i]['img'] if len(tiles) > i \
                       else np.ones((self.image_size, self.image_size, 3), np.uint8) * 255
                tile = 255 - tile
                out[hh*image_size:(hh+1)*image_size, ww*image_size:(ww+1)*image_size] = tile
        out = (out.astype(np.float32) / 255).transpose(2, 0, 1)
        return torch.tensor(out)

### Preview a few test montages

In [ ]:
preview = PANDADataset(df, image_size, n_tiles, 0)
fig, ax = plt.subplots(1, 4, figsize=(18, 5))
for p in range(4):
    img = preview[p]
    ax[p].imshow(1. - img.permute(1, 2, 0).numpy()); ax[p].axis('off')
    ax[p].set_title(df.image_id.iloc[p][:10])
plt.suptitle('Test montages (display un-inverted)'); plt.show()

## Prediction with 2× TTA

In [ ]:
def predict(tile_mode):
    loader = DataLoader(PANDADataset(df, image_size, n_tiles, tile_mode),
                        batch_size=batch_size, num_workers=num_workers, shuffle=False)
    out = []
    with torch.no_grad():
        for data in tqdm(loader, desc=f'mode {tile_mode}'):
            out.append(model(data.to(device)).sigmoid().cpu())
    return torch.cat(out)

probs = (predict(0) + predict(2)) / 2
preds = probs.sum(1).round().clamp(0, 5).numpy().astype(int)
df['isup_grade'] = preds
df[['image_id', 'isup_grade']].to_csv('submission.csv', index=False)
print(df[['image_id', 'isup_grade']].head())

## Prediction summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pd.Series(preds).value_counts().sort_index().plot.bar(ax=ax[0], color='#4C72B0')
ax[0].set_title('Predicted ISUP distribution'); ax[0].set_xlabel('ISUP'); ax[0].set_ylabel('# slides')
ax[1].imshow(1. - preview[0].permute(1, 2, 0).numpy()); ax[1].axis('off')
ax[1].set_title(f'{df.image_id.iloc[0][:12]} -> predicted ISUP {preds[0]}')
plt.tight_layout(); plt.show()